In [ ]:
%%bash
# ASL Citizen landmark extraction, part 3 of 3 (edge-vision).
# Settings: Accelerator None (CPU), Internet On. Run with Save Version > Save & Run All (Commit).
# Inputs: Your Work > Datasets > edge-vision-src.
# Tracks every 3rd shard (EV_PART 2/3); run the 3 part notebooks side by side, then
# asl_citizen_merge. Up to 11 hours. If the log says INCOMPLETE, add this notebook's own output
# as an input and run it again; finished shards are skipped.
# Licence: ASL Citizen is non-commercial research only; never redistribute it or the landmarks,
# keep this notebook and its outputs private, delete everything when the research is complete
# (docs/DATA_LICENSES.md). Generated by scripts/build_kaggle_notebooks.py; edit the sources.
set -u
export EV_LIMIT_VIDEOS=0    # > 0: SMOKE TEST of the first N videos, into a separate -smoke folder
export EV_PART="2/3"         # k/N: this run's share of the shards
NAME=asl_citizen_part3
# ---- common to the edge-vision Kaggle notebooks (scripts/build_kaggle_notebooks.py) ----
# EV_WORK, EV_INPUT and EV_TMP exist for local tests only (scripts/kaggle_local_test.py).
WORK="${EV_WORK:-/kaggle/working}"
export EV_INPUT="${EV_INPUT:-/kaggle/input}"
TMPROOT="${EV_TMP:-/tmp}"
SRC="$TMPROOT/edge-vision-src"
# The fingerprint of the sources this notebook was built from; the code archive must match it.
SOURCE_FINGERPRINT="15904537e7a76f5a"
VENV_PYTHON="3.11"
mkdir -p "$WORK" "$TMPROOT"
# clean runs a venv's Python with only the variables it needs. The Kaggle kernel exports its
# own: PYTHONPATH holds a sitecustomize that imports wrapt, and MPLBACKEND names
# matplotlib_inline, which made "import mediapipe" (it imports matplotlib.pyplot) fail in the
# venv. So the environment starts empty, keeps CLEAN_KEEP and every EV_* variable, and sets
# MPLBACKEND=Agg.
CLEAN_KEEP="HOME USER LOGNAME PATH LANG LC_ALL LC_CTYPE TZ TMPDIR TEMP TMP TERM
  LD_LIBRARY_PATH CUDA_VISIBLE_DEVICES CUDA_HOME NVIDIA_VISIBLE_DEVICES NVIDIA_DRIVER_CAPABILITIES
  OMP_NUM_THREADS HTTP_PROXY HTTPS_PROXY NO_PROXY http_proxy https_proxy no_proxy
  SSL_CERT_FILE SSL_CERT_DIR SYSTEMROOT WINDIR COMSPEC USERPROFILE APPDATA LOCALAPPDATA PATHEXT
  NUMBER_OF_PROCESSORS PROCESSOR_ARCHITECTURE"
clean() {
  local name keep=()
  for name in $CLEAN_KEEP $(compgen -e EV_); do
    if [ -n "${!name+x}" ]; then keep+=("$name=${!name}"); fi
  done
  env -i "${keep[@]}" MPLBACKEND=Agg PYTHONNOUSERSITE=1 "$@"
}
# Kaggle also sets UV_SYSTEM_PYTHON ("The --system flag has no effect" in uv venv's output).
uv() { env -u UV_SYSTEM_PYTHON -u VIRTUAL_ENV python -m uv "$@"; }
venv_python() { if [ -e "$1/bin/python" ]; then echo "$1/bin/python"; else echo "$1/Scripts/python.exe"; fi; }
kaggle_python() { python -c 'import platform; print(platform.python_version())' 2>&1; }
# find below the inputs without walking the asl-signs parquet tree or the landmark shards
find_inputs() { find "$EV_INPUT" \( -name train_landmarks -o -name shards \) -prune -o "$@" -print 2>/dev/null | sort; }

# make_venv DIR: an empty Python $VENV_PYTHON venv, made with uv (Kaggle's Python has no
# ensurepip, and no mediapipe 0.10.21 wheel exists for it).
make_venv() {
  echo "--- pip install uv"
  python -m uv --version 2>/dev/null || python -m pip install -q --disable-pip-version-check uv \
    || { echo "STEP FAILED: pip install uv"; return 1; }
  echo "--- uv python install $VENV_PYTHON"
  uv python install "$VENV_PYTHON" || { echo "STEP FAILED: uv python install $VENV_PYTHON"; return 1; }
  echo "--- uv venv --python $VENV_PYTHON $1"
  uv venv --clear --python "$VENV_PYTHON" "$1" || { echo "STEP FAILED: uv venv $1"; return 1; }
}

# The code comes ONLY from the private dataset edge-vision-src: edge-vision-src.tar.gz, made by
# "python scripts/build_kaggle_notebooks.py --source-archive", or that archive unpacked by Kaggle.
get_source() {
  local archives archive tree found
  rm -rf "$SRC"
  archives=$(find_inputs -type f -name 'edge-vision-src*.tar.gz')
  archive=$(echo "$archives" | head -1)
  tree=$(find_inputs -type f -name SOURCE_COMMIT | head -1)
  if [ "$(echo "$archives" | grep -c .)" -gt 1 ]; then
    echo "WARNING: several code archives are attached; using the first:"
    echo "$archives"
  fi
  if [ -n "$archive" ]; then
    echo "code: $archive"
    python -c 'import sys, tarfile; tarfile.open(sys.argv[1]).extractall(sys.argv[2], filter="data")' \
      "$archive" "$SRC" || { echo "CODE SOURCE UNREADABLE: could not unpack $archive"; return 1; }
  elif [ -n "$tree" ]; then
    echo "code: $(dirname "$tree") (the archive, unpacked by Kaggle)"
    cp -r "$(dirname "$tree")" "$SRC" \
      || { echo "CODE SOURCE UNREADABLE: could not copy $(dirname "$tree")"; return 1; }
  else
    echo "CODE SOURCE MISSING: no edge-vision-src.tar.gz under $EV_INPUT."
    echo "  These notebooks never clone the repository. Upload artifacts/v3/upload/edge-vision-src.tar.gz"
    echo "  (python scripts/build_kaggle_notebooks.py --source-archive) as a new Private dataset named"
    echo "  edge-vision-src, add it here with Add Input > Your Work > Datasets, and run again."
    return 1
  fi
  if [ ! -f "$SRC/SOURCE_COMMIT" ]; then
    echo "CODE SOURCE INVALID: no SOURCE_COMMIT in it; build the archive with --source-archive"
    return 1
  fi
  echo "code commit: $(cat "$SRC/SOURCE_COMMIT")"
  found=$(python "$SRC/scripts/build_kaggle_notebooks.py" --fingerprint 2>&1)
  if [ "$found" != "$SOURCE_FINGERPRINT" ]; then
    echo "CODE SOURCE MISMATCH: this notebook was built from notebook sources $SOURCE_FINGERPRINT,"
    echo "  the edge-vision-src input holds '$found'. Use the archive and the notebooks of one commit:"
    echo "  upload that commit's edge-vision-src.tar.gz as a new version of the dataset, or import"
    echo "  the notebooks again from the commit the archive was made from."
    return 1
  fi
  echo "code fingerprint: $found (matches this notebook)"
}
# ---- end of the common part ----
MPENV="$TMPROOT/mpenv"
venv_versions() {
  clean "$(venv_python "$MPENV")" -c 'import platform, mediapipe, cv2, numpy; print(f"venv python {platform.python_version()}: mediapipe {mediapipe.__version__}, cv2 {cv2.__version__}, numpy {numpy.__version__}")'
}
setup_environment() {
  local py
  if [ -e "$(venv_python "$MPENV")" ] && venv_versions 2>/dev/null; then
    echo "reusing $MPENV"
    return 0
  fi
  make_venv "$MPENV" || return 1
  py=$(venv_python "$MPENV")
  echo '--- uv pip install mediapipe==0.10.21 opencv-python-headless "numpy<2" pandas pyarrow'
  uv pip install --python "$py" mediapipe==0.10.21 opencv-python-headless "numpy<2" pandas pyarrow || { echo "STEP FAILED: uv pip install"; return 1; }
  # mediapipe also requires opencv-contrib-python, which writes the same cv2 package as the
  # headless build. Keep exactly one cv2: drop contrib, then restore the headless files it shared.
  if uv pip show --python "$py" opencv-contrib-python >/dev/null 2>&1; then
    OPENCV=$(uv pip show --python "$py" opencv-python-headless | sed -n 's/^Version: //p')
    echo "--- keeping only opencv-python-headless==$OPENCV"
    uv pip uninstall --python "$py" opencv-contrib-python \
      && uv pip install --python "$py" --reinstall "opencv-python-headless==$OPENCV" "numpy<2" \
      || { echo "STEP FAILED: replacing opencv-contrib-python with the headless build"; return 1; }
  fi
  echo "--- versions inside $MPENV"
  venv_versions || { echo "STEP FAILED: importing mediapipe, cv2 and numpy in $MPENV"; return 1; }
}
if [ "$EV_LIMIT_VIDEOS" -gt 0 ]; then
  OUT="$WORK/asl-citizen-landmarks-smoke"
else
  OUT="$WORK/asl-citizen-landmarks"
fi
export EV_OUT="$OUT"
LOG="$WORK/$NAME.log"
finish() {
  [ -n "${1:-}" ] && echo "$1" | tee -a "$LOG"
  echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0, so read $LOG" | tee -a "$LOG"
  exit 0
}
echo "=== $(date -u '+%F %T') UTC: start $NAME (EV_PART $EV_PART, EV_LIMIT_VIDEOS $EV_LIMIT_VIDEOS); Kaggle python $(kaggle_python)" | tee -a "$LOG"
get_source 2>&1 | tee -a "$LOG"
[ "${PIPESTATUS[0]}" -eq 0 ] || finish "STOPPED: no usable code source (see above); nothing was extracted."

setup_environment 2>&1 | tee -a "$LOG"
[ "${PIPESTATUS[0]}" -eq 0 ] || finish "ENVIRONMENT FAILED: nothing was extracted. The full error is above and in $LOG."
PY=$(venv_python "$MPENV")

echo "--- the extraction program, built from the code source" | tee -a "$LOG"
rm -f "$WORK/$NAME.py"
python "$SRC/scripts/build_kaggle_notebooks.py" --script-out "$WORK/$NAME.py" 2>&1 | tee -a "$LOG"
[ -f "$WORK/$NAME.py" ] || finish "STOPPED: the extraction program could not be built (see above)."
clean "$PY" -u "$WORK/$NAME.py" 2>&1 | tee -a "$LOG"
echo "=== extraction exit status ${PIPESTATUS[0]}" | tee -a "$LOG"
finish
